In [5]:
"""
CSE715 Neural Networks — Hard Task (Real Data Version)
=======================================================
Conditional VAE (CVAE) + Beta-VAE for Disentangled Latent Representations
Multi-modal clustering: MFCC audio + Char-ngram TF-IDF lyrics + Genre one-hot

Data sources
  • English lyrics : sample_100k_lyrics.csv          (5 000 → 500 sampled)
  • Bangla  lyrics : BanglaSongLyrics.csv             (4 105 → 500 sampled)
  • Audio   (MP3)  : million-song-dataset /MP3-Example (1 500 → 300 sampled)

Feature pipeline
  • Audio   : librosa MFCC (40 coefficients, mean over time)  → 40-d
  • Lyrics  : sklearn TfidfVectorizer(analyzer='char_wb',      → 100-d
              ngram_range=(2,4)) → SVD → 100-d dense
  • Genre   : Label-encoded from audio_df genre column        → one-hot 5-d
              (used as CVAE condition and as ground-truth label)
  • Language: 0=English, 1=Bangla                             → label only

Fusion
  • Multimodal input  = [audio(40) | lyrics(100)]  → 140-d
  • CVAE condition    = genre one-hot (5-d)
"""

# ──────────────────────────────────────────────────────────────────────────────
# 0.  IMPORTS
# ──────────────────────────────────────────────────────────────────────────────
import os, warnings, time
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.cluster       import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics       import (silhouette_score,
                                   adjusted_rand_score,
                                   normalized_mutual_info_score,
                                   confusion_matrix)
from sklearn.manifold      import TSNE

import librosa          # pip install librosa  (available on Kaggle by default)

np.random.seed(42)

# ──────────────────────────────────────────────────────────────────────────────
# 1.  CONFIGURATION  ← only section you need to edit
# ──────────────────────────────────────────────────────────────────────────────
CFG = dict(
    # ── Kaggle paths ──────────────────────────────────────────────────────
    english_lyrics_csv = '/kaggle/input/datasets/mdtamim26301/sample/sample_10k_lyrics.csv',
    bangla_lyrics_csv  = '/kaggle/input/datasets/shakirulhasan/bangla-song-lyrics/BanglaSongLyrics.csv',
    audio_base_path    = '/kaggle/input/datasets/undefinenull/million-song-dataset-spotify-lastfm/MP3-Example',

    # ── Sample sizes ──────────────────────────────────────────────────────
    n_english   = 500,    # English lyrics tracks to use
    n_bangla    = 500,    # Bangla  lyrics tracks to use
    n_audio_per_genre = 60,   # audio clips per genre  (5 genres × 60 = 300)

    # ── Feature dimensions ────────────────────────────────────────────────
    n_mfcc      = 40,     # MFCC coefficients per clip
    lyrics_svd  = 100,    # SVD dimensions after TF-IDF
    latent_dim  = 16,     # VAE latent space size
    hidden_dim  = 128,    # MLP hidden layer width

    # ── Training ──────────────────────────────────────────────────────────
    epochs      = 250,
    batch_size  = 64,
    lr          = 5e-4,
    beta        = 4.0,    # β for Beta-VAE

    # ── Output ────────────────────────────────────────────────────────────
    output_dir  = '/kaggle/working/',
    random_seed = 42,
)

# ──────────────────────────────────────────────────────────────────────────────
# 2.  DATA LOADING
# ──────────────────────────────────────────────────────────────────────────────

def load_english_lyrics(path, n=500):
    """Load English lyrics from sample_100k CSV, drop empty rows."""
    df = pd.read_csv(path)
    df = df[df['language'] == 'en'].copy()
    # Keep only useful columns; 'tag' is the genre proxy for English
    df = df[['title', 'tag', 'lyrics']].dropna(subset=['lyrics'])
    df = df[df['lyrics'].str.strip().ne('')]
    df = df.sample(n=min(n, len(df)), random_state=CFG['random_seed'])
    df['language'] = 'english'
    df = df.rename(columns={'tag': 'genre_label'})
    print(f"  English lyrics loaded : {len(df)} rows  |  genres: {df['genre_label'].unique()}")
    return df.reset_index(drop=True)


def load_bangla_lyrics(path, n=500):
    """Load Bangla lyrics.  'category' column is the genre proxy."""
    df = pd.read_csv(path)
    df = df[['title', 'category', 'lyrics']].dropna(subset=['lyrics'])
    df = df[df['lyrics'].str.strip().ne('')]
    df = df.sample(n=min(n, len(df)), random_state=CFG['random_seed'])
    df['language'] = 'bangla'
    df = df.rename(columns={'category': 'genre_label'})
    print(f"  Bangla  lyrics loaded : {len(df)} rows  |  genres: {df['genre_label'].unique()}")
    return df.reset_index(drop=True)


def load_audio_features(base_path, n_per_genre=60):
    """
    Walk the MP3-Example directory (genre sub-folders), extract
    40 MFCC coefficients (mean over frames) per file.
    Returns DataFrame with columns: [genre, mfcc_0 … mfcc_39]
    """
    rows = []
    skipped = 0

    for genre in sorted(os.listdir(base_path)):
        genre_path = os.path.join(base_path, genre)
        if not os.path.isdir(genre_path):
            continue

        mp3_files = [f for f in os.listdir(genre_path) if f.endswith('.mp3')]
        # Sample n_per_genre files
        selected = np.random.RandomState(CFG['random_seed']).choice(
            mp3_files, size=min(n_per_genre, len(mp3_files)), replace=False
        )

        print(f"  Extracting MFCC — genre: {genre:<12s}  ({len(selected)} files)", end='', flush=True)
        t0 = time.time()

        for fname in selected:
            fpath = os.path.join(genre_path, fname)
            try:
                y, sr = librosa.load(fpath, sr=22050, mono=True, duration=30)
                mfcc  = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=CFG['n_mfcc'])
                feats = mfcc.mean(axis=1)   # shape (40,)
                rows.append([genre] + feats.tolist())
            except Exception as e:
                skipped += 1

        print(f"  [{time.time()-t0:.1f}s]")

    if skipped:
        print(f"  Skipped {skipped} files due to errors.")

    cols = ['genre'] + [f'mfcc_{i}' for i in range(CFG['n_mfcc'])]
    df   = pd.DataFrame(rows, columns=cols)
    print(f"  Audio features ready  : {len(df)} tracks  |  genres: {df['genre'].unique()}")
    return df.reset_index(drop=True)


# ──────────────────────────────────────────────────────────────────────────────
# 3.  FEATURE ENGINEERING
# ──────────────────────────────────────────────────────────────────────────────

def build_lyrics_features(lyrics_series, svd_dim=100):
    """
    Character n-gram TF-IDF (2,4) works for both Latin (English)
    and Bengali Unicode scripts without any tokenisation.
    Reduce to svd_dim dimensions via TruncatedSVD.
    """
    tfidf = TfidfVectorizer(
        analyzer       = 'char_wb',
        ngram_range    = (2, 4),
        max_features   = 8000,
        sublinear_tf   = True,
        strip_accents  = None,   # keep Unicode chars for Bangla
        min_df         = 2,
    )
    X_sparse = tfidf.fit_transform(lyrics_series.fillna(''))

    svd = TruncatedSVD(n_components=min(svd_dim, X_sparse.shape[1] - 1),
                       random_state=CFG['random_seed'])
    X_dense  = svd.fit_transform(X_sparse)
    explained = svd.explained_variance_ratio_.sum()
    print(f"  TF-IDF SVD: {X_sparse.shape[1]} vocab → {X_dense.shape[1]}d "
          f"(explained var: {explained:.2%})")
    return X_dense, tfidf, svd


def build_genre_onehot(genre_labels):
    """Encode genre strings → integer ids → one-hot matrix."""
    le = LabelEncoder()
    ids = le.fit_transform(genre_labels)
    n_classes = len(le.classes_)
    oh = np.zeros((len(ids), n_classes), dtype=np.float32)
    oh[np.arange(len(ids)), ids] = 1.0
    return oh, ids, le


# ──────────────────────────────────────────────────────────────────────────────
# 4.  DATASET ASSEMBLY
#     We build two aligned sub-datasets, then merge:
#       (A) Audio-centric  — audio_df rows matched to genre labels
#       (B) Lyrics-centric — english + bangla lyrics
#     Final fused set: every row has [audio | lyrics | genre_one_hot]
#
#     Strategy:
#       • audio_df  (300 rows)  gets audio features + genre label
#       • We sample matching lyrics from english/bangla for each audio row
#         by matching on genre (best-effort; genres won't always align)
#       • Rows that can't be genre-matched get lyrics from random same-lang pool
# ──────────────────────────────────────────────────────────────────────────────

def assemble_dataset(audio_df, en_df, bn_df):
    """
    Fuse audio + lyrics into one aligned DataFrame.
    Each audio clip is paired with one lyrics entry of matching genre.
    Language is assigned alternately (English / Bangla) to get balance.
    """
    # ── 1. Normalise genre names ──────────────────────────────────────────
    # Audio genres are title-cased: Metal, Pop, Rock, Classical, Country …
    # Map Bangla categories to closest audio genre for one-hot consistency
    BANGLA_GENRE_MAP = {
        'ব্যান্ড':           'Rock',
        'দেশাত্মবোধক গান':   'Classical',
        'বাউল':              'Folk',
        'নজরুল গীতি':        'Classical',
        'আধুনিক':            'Pop',
        'ছায়াছবি':           'Pop',
        'রবীন্দ্র সঙ্গীত':   'Classical',
        'লোকগীতি':           'Folk',
        'পল্লীগীতি':         'Folk',
        'জারি':              'Folk',
    }
    EN_GENRE_MAP = {
        'pop': 'Pop', 'rock': 'Rock', 'rb': 'R&B',
        'rap': 'Rap', 'misc': 'Pop', 'metal': 'Metal',
        'country': 'Country', 'jazz': 'Jazz',
        'classical': 'Classical', 'folk': 'Folk',
    }

    bn_df = bn_df.copy()
    bn_df['genre_norm'] = bn_df['genre_label'].map(BANGLA_GENRE_MAP).fillna('Pop')
    en_df = en_df.copy()
    en_df['genre_norm'] = en_df['genre_label'].str.lower().map(EN_GENRE_MAP).fillna('Pop')

    audio_df = audio_df.copy()
    audio_df['genre_norm'] = audio_df['genre'].str.title()

    # ── 2. Build lyrics lookup dicts by genre ────────────────────────────
    en_by_genre = {g: grp for g, grp in en_df.groupby('genre_norm')}
    bn_by_genre = {g: grp for g, grp in bn_df.groupby('genre_norm')}

    def sample_lyric(genre, pool_dict, fallback_df, rng):
        pool = pool_dict.get(genre, fallback_df)
        if len(pool) == 0:
            pool = fallback_df
        return pool.sample(1, random_state=rng).iloc[0]['lyrics']

    # ── 3. Pair each audio clip with a lyric ─────────────────────────────
    fused_rows = []
    rng_state  = 0
    for idx, row in audio_df.iterrows():
        genre = row['genre_norm']
        lang  = 'english' if (idx % 2 == 0) else 'bangla'
        pool_dict = en_by_genre if lang == 'english' else bn_by_genre
        fallback  = en_df       if lang == 'english' else bn_df

        lyric = sample_lyric(genre, pool_dict, fallback, rng_state)
        rng_state += 1

        mfcc_cols = [row[f'mfcc_{i}'] for i in range(CFG['n_mfcc'])]
        fused_rows.append({
            'genre'   : genre,
            'language': lang,
            'lyrics'  : lyric,
            **{f'mfcc_{i}': mfcc_cols[i] for i in range(CFG['n_mfcc'])},
        })

    fused_df = pd.DataFrame(fused_rows).reset_index(drop=True)
    print(f"\n  Fused dataset: {len(fused_df)} rows")
    print(f"  Genre distribution:\n{fused_df['genre'].value_counts().to_string()}")
    print(f"  Language distribution:\n{fused_df['language'].value_counts().to_string()}")
    return fused_df


# ──────────────────────────────────────────────────────────────────────────────
# 5.  NUMPY VAE / BETA-VAE / CVAE  (unchanged from synthetic version)
# ──────────────────────────────────────────────────────────────────────────────

def relu(x):      return np.maximum(0, x)
def relu_grad(x): return (x > 0).astype(float)

class AdamOptimizer:
    def __init__(self, lr=1e-3, beta1=0.9, beta2=0.999, eps=1e-8):
        self.lr=lr; self.b1=beta1; self.b2=beta2; self.eps=eps
        self.m={}; self.v={}; self.t=0

    def update(self, params, grads):
        self.t += 1
        for k in params:
            if k not in self.m:
                self.m[k] = np.zeros_like(params[k])
                self.v[k] = np.zeros_like(params[k])
            self.m[k] = self.b1*self.m[k] + (1-self.b1)*grads[k]
            self.v[k] = self.b2*self.v[k] + (1-self.b2)*grads[k]**2
            mh = self.m[k]/(1-self.b1**self.t)
            vh = self.v[k]/(1-self.b2**self.t)
            params[k] -= self.lr*mh/(np.sqrt(vh)+self.eps)
        return params


def _init(i, o, s=0.01):
    return np.random.randn(i, o)*s, np.zeros(o)


class NumpyVAE:
    """
    Vanilla VAE (beta=1), Beta-VAE (beta>1), CVAE (condition_dim>0).
    Pure NumPy MLP with Adam + reparameterisation trick.
    """
    def __init__(self, input_dim, latent_dim=16, hidden_dim=128,
                 beta=1.0, condition_dim=0, lr=5e-4):
        self.input_dim     = input_dim
        self.latent_dim    = latent_dim
        self.hidden_dim    = hidden_dim
        self.beta          = beta
        self.condition_dim = condition_dim
        enc_in = input_dim + condition_dim
        dec_in = latent_dim + condition_dim

        We1,be1 = _init(enc_in,     hidden_dim)
        We2,be2 = _init(hidden_dim, hidden_dim)
        Wmu,bmu = _init(hidden_dim, latent_dim)
        Wlv,blv = _init(hidden_dim, latent_dim)
        Wd1,bd1 = _init(dec_in,     hidden_dim)
        Wd2,bd2 = _init(hidden_dim, hidden_dim)
        Wo, bo  = _init(hidden_dim, input_dim)

        self.params = dict(We1=We1,be1=be1,We2=We2,be2=be2,
                           Wmu=Wmu,bmu=bmu,Wlv=Wlv,blv=blv,
                           Wd1=Wd1,bd1=bd1,Wd2=Wd2,bd2=bd2,Wo=Wo,bo=bo)
        self.opt    = AdamOptimizer(lr=lr)
        self.losses = []

    # ── forward helpers ──────────────────────────────────────────────────
    def _enc(self, X, c=None):
        inp = np.hstack([X, c]) if c is not None else X
        h1  = relu(inp @ self.params['We1'] + self.params['be1'])
        h2  = relu(h1  @ self.params['We2'] + self.params['be2'])
        mu  = h2 @ self.params['Wmu'] + self.params['bmu']
        lv  = np.clip(h2 @ self.params['Wlv'] + self.params['blv'], -4, 4)
        return mu, lv, h1, h2, inp

    def _dec(self, z, c=None):
        inp = np.hstack([z, c]) if c is not None else z
        h1  = relu(inp @ self.params['Wd1'] + self.params['bd1'])
        h2  = relu(h1  @ self.params['Wd2'] + self.params['bd2'])
        xh  = h2 @ self.params['Wo']  + self.params['bo']
        return xh, h1, h2, inp

    def _reparam(self, mu, lv):
        eps = np.random.randn(*mu.shape)
        return mu + np.exp(0.5*lv)*eps, eps

    # ── training step (manual backprop) ──────────────────────────────────
    def train_step(self, X, c=None):
        N = X.shape[0]
        mu, lv, eh1, eh2, enc_inp = self._enc(X, c)
        z,  eps                   = self._reparam(mu, lv)
        xh, dh1, dh2, dec_inp     = self._dec(z, c)

        recon = np.mean((X - xh)**2)
        kl    = -0.5 * np.mean(1 + lv - mu**2 - np.exp(lv))

        grads = {k: np.zeros_like(v) for k, v in self.params.items()}

        # decoder grads
        d_xh = -2*(X - xh)/N
        grads['Wo']  = dh2.T @ d_xh / N
        grads['bo']  = d_xh.mean(0)
        d_dh2 = d_xh @ self.params['Wo'].T * relu_grad(dh2)
        grads['Wd2'] = dh1.T @ d_dh2 / N
        grads['bd2'] = d_dh2.mean(0)
        d_dh1 = d_dh2 @ self.params['Wd2'].T * relu_grad(dh1)
        grads['Wd1'] = dec_inp.T @ d_dh1 / N
        grads['bd1'] = d_dh1.mean(0)
        d_dec_inp = d_dh1 @ self.params['Wd1'].T
        d_z = d_dec_inp[:, :self.latent_dim]

        # encoder + KL grads
        d_mu = self.beta * mu / N + d_z
        d_lv = self.beta * 0.5*(np.exp(lv)-1)/N + d_z*(0.5*np.exp(0.5*lv)*eps)
        grads['Wmu'] = eh2.T @ d_mu / N; grads['bmu'] = d_mu.mean(0)
        grads['Wlv'] = eh2.T @ d_lv / N; grads['blv'] = d_lv.mean(0)
        d_eh2 = (d_mu @ self.params['Wmu'].T + d_lv @ self.params['Wlv'].T)*relu_grad(eh2)
        grads['We2'] = eh1.T @ d_eh2 / N; grads['be2'] = d_eh2.mean(0)
        d_eh1 = d_eh2 @ self.params['We2'].T * relu_grad(eh1)
        grads['We1'] = enc_inp.T @ d_eh1 / N; grads['be1'] = d_eh1.mean(0)

        self.params = self.opt.update(self.params, grads)
        return recon + self.beta*kl, recon, kl

    def fit(self, X, c=None, epochs=250, batch_size=64, verbose=True):
        N = X.shape[0]
        for ep in range(1, epochs+1):
            idx = np.random.permutation(N)
            ep_loss = 0.0
            for i in range(0, N, batch_size):
                b  = idx[i:i+batch_size]
                cb = c[b] if c is not None else None
                tl,_,_ = self.train_step(X[b], cb)
                ep_loss += tl
            self.losses.append(ep_loss)
            if verbose and ep % 50 == 0:
                print(f"    Epoch {ep:>3d}/{epochs}  loss={ep_loss:.4f}")

    def get_latent(self, X, c=None):
        mu,_,_,_,_ = self._enc(X, c)
        return mu

    def reconstruct(self, X, c=None):
        mu,_,_,_,_ = self._enc(X, c)
        xh,_,_,_   = self._dec(mu, c)
        return xh


class ShallowAE:
    def __init__(self, input_dim, latent_dim=16, hidden_dim=128, lr=5e-4):
        self.params = {}
        pairs = [('W1',input_dim,hidden_dim),('W2',hidden_dim,latent_dim),
                 ('W3',latent_dim,hidden_dim),('W4',hidden_dim,input_dim)]
        for name,i,o in pairs:
            self.params[name], self.params[name.replace('W','b')] = _init(i,o)
        self.opt = AdamOptimizer(lr=lr)

    def forward(self, X):
        h1 = relu(X  @ self.params['W1'] + self.params['b1'])
        z  = relu(h1 @ self.params['W2'] + self.params['b2'])
        h2 = relu(z  @ self.params['W3'] + self.params['b3'])
        xh = h2 @ self.params['W4'] + self.params['b4']
        return xh, z, h1, h2

    def fit(self, X, epochs=250, batch_size=64):
        N = X.shape[0]
        for ep in range(epochs):
            idx = np.random.permutation(N)
            for i in range(0, N, batch_size):
                b = idx[i:i+batch_size]; xb = X[b]; n = len(b)
                xh, z, h1, h2 = self.forward(xb)
                d_xh = -2*(xb-xh)/n; grads = {}
                grads['W4'] = h2.T@d_xh/n; grads['b4'] = d_xh.mean(0)
                d2 = d_xh@self.params['W4'].T*relu_grad(h2)
                grads['W3'] = z.T@d2/n;  grads['b3'] = d2.mean(0)
                dz = d2@self.params['W3'].T*relu_grad(z)
                grads['W2'] = h1.T@dz/n; grads['b2'] = dz.mean(0)
                d1 = dz@self.params['W2'].T*relu_grad(h1)
                grads['W1'] = xb.T@d1/n; grads['b1'] = d1.mean(0)
                self.params = self.opt.update(self.params, grads)

    def get_latent(self, X):
        _, z, _, _ = self.forward(X)
        return z


# ──────────────────────────────────────────────────────────────────────────────
# 6.  CLUSTERING & METRICS
# ──────────────────────────────────────────────────────────────────────────────

def cluster_purity(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    return cm.max(axis=0).sum() / len(y_true)


def evaluate(Z, labels_true, n_clusters, method='kmeans'):
    if method == 'kmeans':
        m = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    else:
        m = AgglomerativeClustering(n_clusters=n_clusters)
    pred = m.fit_predict(Z)
    return {
        'Silhouette': silhouette_score(Z, pred),
        'NMI'       : normalized_mutual_info_score(labels_true, pred),
        'ARI'       : adjusted_rand_score(labels_true, pred),
        'Purity'    : cluster_purity(labels_true, pred),
        'pred'      : pred,
    }


def tsne2d(Z, perplexity=30):
    return TSNE(n_components=2, perplexity=perplexity,
                random_state=42, max_iter=1000,
                learning_rate='auto', init='pca').fit_transform(Z)


# ──────────────────────────────────────────────────────────────────────────────
# 7.  MAIN PIPELINE
# ──────────────────────────────────────────────────────────────────────────────

print("=" * 65)
print("  CSE715 Hard Task — Real Data Pipeline")
print("=" * 65)

# ── 7.1  Load raw data ───────────────────────────────────────────────────────
print("\n[1/7] Loading raw data …")
en_df    = load_english_lyrics(CFG['english_lyrics_csv'], CFG['n_english'])
bn_df    = load_bangla_lyrics (CFG['bangla_lyrics_csv'],  CFG['n_bangla'])
audio_df = load_audio_features(CFG['audio_base_path'],    CFG['n_audio_per_genre'])

# ── 7.2  Assemble fused dataset ──────────────────────────────────────────────
print("\n[2/7] Assembling fused dataset …")
fused = assemble_dataset(audio_df, en_df, bn_df)
N     = len(fused)

# ── 7.3  Feature engineering ────────────────────────────────────────────────
print("\n[3/7] Extracting features …")

# Audio: already extracted as mfcc_0 … mfcc_39
mfcc_cols = [f'mfcc_{i}' for i in range(CFG['n_mfcc'])]
X_audio   = fused[mfcc_cols].values.astype(np.float32)

# Lyrics: char-ngram TF-IDF → SVD
print("  Building lyrics features (char 2-4 ngram TF-IDF → SVD) …")
X_lyrics_dense, tfidf_model, svd_model = build_lyrics_features(
    fused['lyrics'], svd_dim=CFG['lyrics_svd']
)
X_lyrics = X_lyrics_dense.astype(np.float32)

# Genre one-hot (from audio genre column)
genre_oh, genre_ids, genre_le = build_genre_onehot(fused['genre'])
lang_ids = LabelEncoder().fit_transform(fused['language'])
N_GENRE  = genre_oh.shape[1]
GENRE_NAMES = list(genre_le.classes_)
LANG_NAMES  = ['bangla', 'english']

print(f"  Audio features  : {X_audio.shape}")
print(f"  Lyrics features : {X_lyrics.shape}")
print(f"  Genre one-hot   : {genre_oh.shape}  classes={GENRE_NAMES}")

# Scale
scaler_a = StandardScaler(); X_audio  = scaler_a.fit_transform(X_audio)
scaler_l = StandardScaler(); X_lyrics = scaler_l.fit_transform(X_lyrics)

# Multimodal concat
X_mm  = np.hstack([X_audio, X_lyrics]).astype(np.float32)   # (N, 40+lyrics_svd)
INPUT_DIM = X_mm.shape[1]
N_CLUSTERS = len(GENRE_NAMES)
print(f"\n  Final input dim : {INPUT_DIM}   N_clusters : {N_CLUSTERS}")

# ── 7.4  Train models ────────────────────────────────────────────────────────
print("\n[4/7] Training models …")

print("  (A) Vanilla VAE (β=1) …")
vae = NumpyVAE(INPUT_DIM, CFG['latent_dim'], CFG['hidden_dim'],
               beta=1.0, lr=CFG['lr'])
vae.fit(X_mm, epochs=CFG['epochs'], batch_size=CFG['batch_size'], verbose=True)
Z_vae = vae.get_latent(X_mm)

print("  (B) Beta-VAE (β=4) …")
bvae = NumpyVAE(INPUT_DIM, CFG['latent_dim'], CFG['hidden_dim'],
                beta=CFG['beta'], lr=CFG['lr'])
bvae.fit(X_mm, epochs=CFG['epochs'], batch_size=CFG['batch_size'], verbose=True)
Z_bvae = bvae.get_latent(X_mm)

print("  (C) CVAE (conditioned on genre) …")
cvae = NumpyVAE(INPUT_DIM, CFG['latent_dim'], CFG['hidden_dim'],
                beta=1.0, condition_dim=N_GENRE, lr=CFG['lr'])
cvae.fit(X_mm, c=genre_oh, epochs=CFG['epochs'],
         batch_size=CFG['batch_size'], verbose=True)
Z_cvae = cvae.get_latent(X_mm, c=genre_oh)

print("  (D) Shallow Autoencoder …")
ae = ShallowAE(INPUT_DIM, CFG['latent_dim'], CFG['hidden_dim'], lr=CFG['lr'])
ae.fit(X_mm, epochs=CFG['epochs'], batch_size=CFG['batch_size'])
Z_ae = ae.get_latent(X_mm)

# ── 7.5  Baselines ───────────────────────────────────────────────────────────
print("\n[5/7] Computing baselines …")
Z_pca    = PCA(n_components=CFG['latent_dim'], random_state=42).fit_transform(X_mm)
Z_audio  = X_audio     # direct spectral (MFCC only)

# ── 7.6  Evaluate ────────────────────────────────────────────────────────────
print("\n[6/7] Evaluating clustering …")
methods = {
    'PCA + K-Means'            : Z_pca,
    'AE + K-Means'             : Z_ae,
    'Direct MFCC + K-Means'    : Z_audio,
    'VAE + K-Means'            : Z_vae,
    'Beta-VAE + K-Means'       : Z_bvae,
    'CVAE + K-Means'           : Z_cvae,
}
results = {}
for name, Z in methods.items():
    r = evaluate(Z, genre_ids, N_CLUSTERS)
    results[name] = r
    print(f"  {name:<32s}  Sil={r['Silhouette']:+.3f}  "
          f"NMI={r['NMI']:.3f}  ARI={r['ARI']:.3f}  Purity={r['Purity']:.3f}")

# ── 7.7  t-SNE ───────────────────────────────────────────────────────────────
print("\n[7/7] t-SNE projections …")
T_pca  = tsne2d(Z_pca)
T_vae  = tsne2d(Z_vae)
T_bvae = tsne2d(Z_bvae)
T_cvae = tsne2d(Z_cvae)


# ──────────────────────────────────────────────────────────────────────────────
# 8.  VISUALIZATION
# ──────────────────────────────────────────────────────────────────────────────
 
BG   = '#0f0f1a'
CARD = '#1a1a2e'
PALETTE_L = ['#00e5ff', '#ff6d00']
SHORT     = ['PCA','AE','MFCC','VAE','β-VAE','CVAE']
 
# Build palette dynamically to handle any number of genres
def _make_palette(n):
    import matplotlib.cm as cm, matplotlib.colors as mcolors
    if n <= 10:   cmap = cm.get_cmap('tab10', n)
    elif n <= 20: cmap = cm.get_cmap('tab20', n)
    else:         cmap = cm.get_cmap('hsv', n)
    return [mcolors.to_hex(cmap(i)) for i in range(n)]
 
n_genres_actual = int(genre_ids.max()) + 1
PALETTE_G = _make_palette(n_genres_actual)
N_CLUSTERS = n_genres_actual   # re-sync cluster count
print(f'  Palette built for {n_genres_actual} genres: {GENRE_NAMES}')
 
def mk_scatter(ax, T, color_ids, palette, title, s=22, alpha=0.82):
    for ci in range(int(color_ids.max()) + 1):
        m = color_ids == ci
        c = palette[ci] if ci < len(palette) else '#aaaaaa'
        ax.scatter(T[m, 0], T[m, 1], color=c, s=s, alpha=alpha, linewidths=0)
    ax.set_title(title, color='white', fontsize=9, fontweight='bold')
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_facecolor(CARD)
    for sp in ax.spines.values(): sp.set_edgecolor('#333')



# ── Figure 1: t-SNE latent spaces ───────────────────────────────────────────
fig1, ax1 = plt.subplots(2, 4, figsize=(18, 9))
fig1.patch.set_facecolor(BG)
for ax in ax1.flat: ax.set_facecolor(CARD)

scatter_cfg = [
    (ax1[0,0], T_pca,  genre_ids, PALETTE_G, 'PCA — by Genre'),
    (ax1[0,1], T_vae,  genre_ids, PALETTE_G, 'VAE — by Genre'),
    (ax1[0,2], T_bvae, genre_ids, PALETTE_G, 'Beta-VAE — by Genre'),
    (ax1[0,3], T_cvae, genre_ids, PALETTE_G, 'CVAE — by Genre'),
    (ax1[1,0], T_pca,  lang_ids,  PALETTE_L, 'PCA — by Language'),
    (ax1[1,1], T_vae,  lang_ids,  PALETTE_L, 'VAE — by Language'),
    (ax1[1,2], T_bvae, lang_ids,  PALETTE_L, 'Beta-VAE — by Language'),
    (ax1[1,3], T_cvae, lang_ids,  PALETTE_L, 'CVAE — by Language'),
]
for (ax, T, cids, pal, title) in scatter_cfg:
    mk_scatter(ax, T, cids, pal, title)

# Legends
gp = [mpatches.Patch(color=PALETTE_G[i], label=GENRE_NAMES[i])
      for i in range(len(GENRE_NAMES))]
ax1[0,3].legend(handles=gp, fontsize=7, loc='lower right',
                framealpha=0.3, labelcolor='white')
lp = [mpatches.Patch(color=PALETTE_L[i], label=LANG_NAMES[i]) for i in range(2)]
ax1[1,3].legend(handles=lp, fontsize=8, loc='lower right',
                framealpha=0.3, labelcolor='white')

fig1.suptitle('t-SNE Latent Spaces — Real Music Data (MFCC + Char-ngram Lyrics)',
              color='white', fontsize=13)
plt.tight_layout()
fig1.savefig(os.path.join(CFG['output_dir'], 'fig1_latent_spaces.png'),
             dpi=150, bbox_inches='tight', facecolor=BG)
plt.close()
print("  fig1_latent_spaces.png saved")

# ── Figure 2: Metrics + Cluster distributions ────────────────────────────────
fig2, ax2 = plt.subplots(2, 3, figsize=(18, 10))
fig2.patch.set_facecolor(BG)
for ax in ax2.flat: ax.set_facecolor(CARD)

BAR_COLORS = ['#607d8b','#78909c','#90a4ae','#E63946','#FF9800','#4CAF50']
METRIC_KEYS = ['Silhouette','NMI','ARI','Purity']
for mi, mk in enumerate(METRIC_KEYS):
    ax = ax2.flat[mi]
    vals = [results[n][mk] for n in methods]
    bars = ax.bar(SHORT, vals, color=BAR_COLORS, edgecolor='none', width=0.65)
    ax.set_title(mk, color='white', fontsize=11, fontweight='bold')
    ymax = max(vals) * 1.25 + 0.01
    ax.set_ylim(0, max(ymax, 0.05))
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8, color='white')
    ax.tick_params(colors='#aaa', labelsize=8)
    for sp in ax.spines.values(): sp.set_visible(False)

# Genre–cluster heatmap (CVAE)
ax_hm = ax2[1, 1]
pred_cvae = results['CVAE + K-Means']['pred']
heat = np.zeros((N_CLUSTERS, len(GENRE_NAMES)))
for ci in range(N_CLUSTERS):
    m = pred_cvae == ci
    if m.sum():
        for gi in range(len(GENRE_NAMES)):
            heat[ci, gi] = (genre_ids[m] == gi).sum()
im = ax_hm.imshow(heat[np.argsort(heat.argmax(1))].T,
                  aspect='auto', cmap='YlOrRd')
ax_hm.set_yticks(range(len(GENRE_NAMES)))
ax_hm.set_yticklabels(GENRE_NAMES, fontsize=7, color='#aaa')
ax_hm.set_xlabel('Cluster (sorted by genre)', color='#aaa', fontsize=8)
ax_hm.set_title('CVAE: Cluster–Genre Heatmap', color='white', fontsize=10, fontweight='bold')
plt.colorbar(im, ax=ax_hm, fraction=0.046, pad=0.04)
ax_hm.tick_params(colors='#aaa')

# Language per cluster
ax_lc = ax2[1, 2]
lang_dist = np.zeros((N_CLUSTERS, 2))
for ci in range(N_CLUSTERS):
    m = pred_cvae == ci
    for li in range(2):
        lang_dist[ci, li] = (lang_ids[m] == li).sum()
x = np.arange(N_CLUSTERS)
ax_lc.bar(x-0.2, lang_dist[:,0], 0.4, label=LANG_NAMES[0], color='#00e5ff', alpha=0.85)
ax_lc.bar(x+0.2, lang_dist[:,1], 0.4, label=LANG_NAMES[1], color='#ff6d00', alpha=0.85)
ax_lc.set_xlabel('Cluster', color='#aaa', fontsize=8)
ax_lc.set_ylabel('Count',   color='#aaa', fontsize=8)
ax_lc.set_title('CVAE: Language per Cluster', color='white', fontsize=10, fontweight='bold')
ax_lc.legend(fontsize=8, framealpha=0.3, labelcolor='white')
ax_lc.tick_params(colors='#aaa')
for sp in ax_lc.spines.values(): sp.set_visible(False)

fig2.suptitle('Clustering Metrics & Distributions', color='white', fontsize=13)
plt.tight_layout()
fig2.savefig(os.path.join(CFG['output_dir'], 'fig2_metrics_distributions.png'),
             dpi=150, bbox_inches='tight', facecolor=BG)
plt.close()
print("  fig2_metrics_distributions.png saved")

# ── Figure 3: Detailed analysis ──────────────────────────────────────────────
fig3 = plt.figure(figsize=(18, 11))
fig3.patch.set_facecolor(BG)
gs = gridspec.GridSpec(3, 4, figure=fig3, hspace=0.45, wspace=0.35)

# (a) MFCC reconstruction examples (4 tracks)
for k, idx in enumerate([0, N//4, N//2, 3*N//4]):
    ax = fig3.add_subplot(gs[0, k])
    orig  = X_mm[idx, :CFG['n_mfcc']]
    recon = cvae.reconstruct(X_mm[idx:idx+1], c=genre_oh[idx:idx+1])[0, :CFG['n_mfcc']]
    ax.plot(orig,  color='#00e5ff', lw=1.3, alpha=0.9, label='Original')
    ax.plot(recon, color='#ff6d00', lw=1.3, alpha=0.9, ls='--', label='CVAE Recon')
    g = GENRE_NAMES[genre_ids[idx]]; l = fused.iloc[idx]['language']
    ax.set_title(f'{g} / {l}', color='white', fontsize=8)
    ax.tick_params(colors='#aaa', labelsize=7)
    ax.set_facecolor(CARD)
    for sp in ax.spines.values(): sp.set_edgecolor('#333')
    if k == 0:
        ax.legend(fontsize=7, framealpha=0.3, labelcolor='white')
        ax.set_ylabel('MFCC', color='#aaa', fontsize=8)

# (b) Latent dimension variance — disentanglement
ax_dz = fig3.add_subplot(gs[1, :2])
lv_vae  = Z_vae.var(0)
lv_bvae = Z_bvae.var(0)
dims = np.arange(CFG['latent_dim'])
ax_dz.bar(dims-0.2, lv_vae,  0.4, label='VAE',       color='#2196F3', alpha=0.85)
ax_dz.bar(dims+0.2, lv_bvae, 0.4, label='β-VAE (β=4)', color='#FF9800', alpha=0.85)
ax_dz.set_xlabel('Latent Dimension', color='#aaa')
ax_dz.set_ylabel('Variance', color='#aaa')
ax_dz.set_title('Latent Variance — Disentanglement Signal', color='white',
                fontsize=10, fontweight='bold')
ax_dz.legend(fontsize=9, framealpha=0.3, labelcolor='white')
ax_dz.tick_params(colors='#aaa'); ax_dz.set_facecolor(CARD)
for sp in ax_dz.spines.values(): sp.set_visible(False)

# (c) Training loss curves
ax_ls = fig3.add_subplot(gs[1, 2:])
smooth = lambda x: np.convolve(x, np.ones(8)/8, 'same')
ax_ls.plot(smooth(vae.losses),  color='#00e5ff', lw=1.5, label='VAE')
ax_ls.plot(smooth(bvae.losses), color='#FF9800', lw=1.5, label='β-VAE')
ax_ls.plot(smooth(cvae.losses), color='#4CAF50', lw=1.5, label='CVAE')
ax_ls.set_xlabel('Epoch', color='#aaa'); ax_ls.set_ylabel('Loss', color='#aaa')
ax_ls.set_title('Training Loss Curves', color='white', fontsize=10, fontweight='bold')
ax_ls.legend(fontsize=9, framealpha=0.3, labelcolor='white')
ax_ls.tick_params(colors='#aaa'); ax_ls.set_facecolor(CARD)
for sp in ax_ls.spines.values(): sp.set_visible(False)

# (d) CVAE 2-D latent dim 0 vs 1
ax_2d = fig3.add_subplot(gs[2, :2])
for gi in range(len(GENRE_NAMES)):
    for li in range(2):
        m  = (genre_ids == gi) & (lang_ids == li)
        mk = ['o','s'][li]
        ax_2d.scatter(Z_cvae[m,0], Z_cvae[m,1], color=PALETTE_G[gi],
                      marker=mk, s=22, alpha=0.8)
ax_2d.set_title('CVAE Dim-0 vs Dim-1\n(colour=genre, shape=language)',
                color='white', fontsize=9)
ax_2d.tick_params(colors='#aaa'); ax_2d.set_facecolor(CARD)
legend_g = [mpatches.Patch(color=PALETTE_G[i], label=GENRE_NAMES[i])
            for i in range(len(GENRE_NAMES))]
ax_2d.legend(handles=legend_g, fontsize=7, framealpha=0.3, labelcolor='white', ncol=2)
for sp in ax_2d.spines.values(): sp.set_visible(False)

# (e) Radar chart
ax_rad = fig3.add_subplot(gs[2, 2:], polar=True)
cats   = ['Silhouette','NMI','ARI','Purity']
ncat   = len(cats)
angles = [n/ncat*2*np.pi for n in range(ncat)] + [0]
for name, color, lbl in [
    ('VAE + K-Means',      '#00e5ff','VAE'),
    ('Beta-VAE + K-Means', '#FF9800','β-VAE'),
    ('CVAE + K-Means',     '#4CAF50','CVAE'),
    ('PCA + K-Means',      '#607d8b','PCA'),
]:
    raw  = [results[name][k] for k in cats]
    norm = [(v+0.2)/1.2 for v in raw] + [(raw[0]+0.2)/1.2]
    ax_rad.plot(angles, norm, color=color, lw=2, label=lbl)
    ax_rad.fill(angles, norm, color=color, alpha=0.05)
ax_rad.set_xticks(angles[:-1])
ax_rad.set_xticklabels(cats, size=8, color='white')
ax_rad.set_yticks([0.25,0.5,0.75,1.0])
ax_rad.set_yticklabels(['','','',''], size=6)
ax_rad.set_facecolor(CARD); ax_rad.grid(color='#333', lw=0.5)
ax_rad.set_title('Metric Radar', color='white', fontsize=10, pad=15)
ax_rad.legend(fontsize=8, loc='upper right',
              bbox_to_anchor=(1.35,1.1), framealpha=0.3, labelcolor='white')

fig3.suptitle('Detailed Model Analysis — Real Data', color='white', fontsize=14, y=1.01)
fig3.savefig(os.path.join(CFG['output_dir'], 'fig3_detailed_analysis.png'),
             dpi=150, bbox_inches='tight', facecolor=BG)
plt.close()
print("  fig3_detailed_analysis.png saved")

# ──────────────────────────────────────────────────────────────────────────────
# 9.  FINAL TABLE
# ──────────────────────────────────────────────────────────────────────────────
print("\n" + "="*65)
print("  FINAL RESULTS — REAL DATA")
print("="*65)
print(f"{'Method':<34s} {'Sil':>7s} {'NMI':>7s} {'ARI':>7s} {'Purity':>7s}")
print("-"*65)
best_nmi = max(results, key=lambda x: results[x]['NMI'])
for name in methods:
    r = results[name]
    tag = " ◄" if name == best_nmi else ""
    print(f"{name:<34s} {r['Silhouette']:>7.3f} {r['NMI']:>7.3f} "
          f"{r['ARI']:>7.3f} {r['Purity']:>7.3f}{tag}")
print("="*65)
print(f"\nBeta-VAE disentanglement ratio (max/min latent var): "
      f"{lv_bvae.max()/max(lv_bvae.min(),1e-9):.1f}x")
print("\nAll figures saved to:", CFG['output_dir'])

  CSE715 Hard Task — Real Data Pipeline

[1/7] Loading raw data …
  English lyrics loaded : 500 rows  |  genres: ['rb' 'rock' 'rap' 'misc' 'pop' 'country']
  Bangla  lyrics loaded : 500 rows  |  genres: ['ব্যান্ড' 'আধুনিক' 'ছায়াছবি' 'রবীন্দ্র সংগীত' 'ইসলামী গান' 'নাটক'
 'নজরুল গীতি' 'বাউল' 'পপ সঙ্গীত' 'বিবিধ' 'পল্লীগীতি' 'দেশাত্মবোধক গান'
 'লালন' 'ভক্তিমূলক গান' 'র\u200d্যাপ' 'জীবনমুখী গান' 'Uncategorized'
 'ছড়াগান' 'ভাটিয়ালি']
  Extracting MFCC — genre: Blues         (60 files)  [7.6s]
  Extracting MFCC — genre: Country       (60 files)  [7.7s]
  Extracting MFCC — genre: Electronic    (60 files)  [7.8s]
  Extracting MFCC — genre: Folk          (60 files)  [7.8s]
  Extracting MFCC — genre: Jazz          (60 files)  [7.8s]
  Extracting MFCC — genre: Latin         (60 files)  [7.6s]
  Extracting MFCC — genre: Metal         (60 files)  [7.7s]
  Extracting MFCC — genre: New Age       (60 files)  [7.7s]
  Extracting MFCC — genre: Pop           (60 files)  [8.0s]
  Extracting MFCC — genre: P